In [1]:
import numpy as np
import xarray as xr
from minisom import MiniSom
import pandas as pd
from sklearn.preprocessing import RobustScaler
import pprint
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.gridspec import GridSpec
import matplotlib.colors as mcolors
from matplotlib import colormaps as cm
from som_2var_training import read_and_transform, build_scaler, train_som

/home/z2034747/.local/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [5]:
prefs = {'filename': "/home/scratch/peldridge/combine_z500_pwat.nc", 
         'var1': 'gh', 'var2': 'pwat', 'wlon': 220, 'elon': 305, 'nlat': 55, 'slat': 20, 
         'som_config': {'x': 5, 'y': 4, 'sigma': 0.5, 'random_seed': 42},
         'som_train': {'num_iteration': 10000, 'random_order': True, 'verbose': True}}

trained_som, trained_scaler, train_info, xr_data, scaled_npy = train_som(prefs)

current model configuration
{'input_len': 24282, 'random_seed': 42, 'sigma': 0.5, 'x': 5, 'y': 4}
current training configuration
{'data': array([[-0.66050106, -0.68316907, -0.70958024, ..., -0.7250001 ,
        -0.7583334 , -0.7583332 ],
       [-0.71175027, -0.7709796 , -0.83132076, ..., -0.39166674,
        -0.35000005, -0.2916665 ],
       [-0.28915408, -0.28948432, -0.29526958, ..., -0.6833334 ,
        -0.8833334 , -1.0499998 ],
       ...,
       [-0.87913156, -0.8735199 , -0.8690234 , ...,  0.5749998 ,
         0.49166647,  0.375     ],
       [-0.31367952, -0.29052007, -0.26362565, ...,  0.25887743,
         0.21721077,  0.16721089],
       [-0.43595633, -0.43200427, -0.43102336, ..., -0.10833359,
        -0.19166677, -0.24999984]], dtype=float32),
 'num_iteration': 10000,
 'random_order': True,
 'verbose': True}
 [ 10000 / 10000 ] 100% - 0:00:00 left 
 quantization error: 73.85657159964673


In [ ]:
win_map = trained_som.win_map(train_info['som_train']['data'])
node_keys = sorted(win_map.keys())

z500_avg_data = [
    np.reshape(
        trained_scaler.inverse_transform([np.mean(win_map[node], axis=0)])[0][:12141],
        (len(xr_data.lat), len(xr_data.lon))
    )
    for node in sorted(win_map.keys())
]

z500_frequencies = trained_som.activation_response(train_info['som_train']['data']).flatten()


pwat_avg_data = [
    np.reshape(
        trained_scaler.inverse_transform([np.mean(win_map[node], axis=0)])[0][12141:],
        (len(xr_data.lat), len(xr_data.lon))
    )
    for node in sorted(win_map.keys())
]

pwat_frequencies = trained_som.activation_response(train_info['som_train']['data']).flatten()


length, width = prefs['som_config']['x'], prefs['som_config']['y']